# Hotel Booking Demand Analysis & Machine Learning Prediction
**Author**: Data Science & Hospitality Analytics Suite  
**Dataset**: 119,390 records across City Hotel and Resort Hotel (2015 - 2017)  
**Objectives**:
1. Exploratory Data Analysis (EDA) uncovering key cancellation drivers, seasonality, and pricing patterns.
2. Machine Learning pipeline to predict booking cancellations (`is_canceled`) with zero data leakage.
3. Operational recommendations and risk mitigation strategies.

## 1. Environment & Data Ingestion

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Load raw dataset
df = pd.read_csv('hotel_bookings.csv')
print(f"Raw dataset shape: {df.shape}")
df.head()

## 2. Data Cleaning & Integrity Safeguards
We identify and resolve:
- Missing values in `children`, `country`, `agent`, and `company`.
- Anomaly entries where total guests (`adults + children + babies`) equals 0.
- ADR (Average Daily Rate) outliers (negative values and extreme entries > $1,000).

In [ ]:
from src.data_pipeline import clean_data, engineer_features

print(f"Null values before cleaning:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

df_clean = clean_data(df)
df_featured = engineer_features(df_clean)
print(f"\nCleaned & Featured dataset shape: {df_featured.shape}")
print(f"Overall Cancellation Rate: {df_featured['is_canceled'].mean():.2%}")

## 3. Exploratory Data Analysis: Key Insights
### 3.1 Hotel Type Cancellation Dynamics

In [ ]:
hotel_summary = df_featured.groupby('hotel').agg(
    total_bookings=('is_canceled', 'count'),
    cancellations=('is_canceled', 'sum'),
    cancellation_rate=('is_canceled', lambda x: round(x.mean() * 100, 2)),
    avg_adr=('adr', lambda x: round(x.mean(), 2)),
    avg_lead_time=('lead_time', lambda x: round(x.mean(), 1))
)
display(hotel_summary)

# Visualization
ax = hotel_summary['cancellation_rate'].plot(kind='bar', color=['#2563eb', '#059669'], rot=0)
plt.title('Cancellation Rate by Hotel Type', fontsize=14, fontweight='bold')
plt.ylabel('Cancellation Rate (%)')
for p in ax.patches:
    ax.annotate(f"{p.get_height()}%", (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', color='white', fontweight='bold', fontsize=12)
plt.show()

### 3.2 Lead Time Horizon vs. Cancellation Probability

In [ ]:
bins = [0, 7, 30, 90, 180, 365, 800]
labels = ['0-7 days', '8-30 days', '1-3 months', '3-6 months', '6-12 months', '1+ year']
df_featured['lead_time_bucket'] = pd.cut(df_featured['lead_time'], bins=bins, labels=labels, include_lowest=True)

lt_rate = df_featured.groupby('lead_time_bucket', observed=False)['is_canceled'].mean() * 100
ax = lt_rate.plot(kind='bar', color='#6366f1', edgecolor='#4338ca', rot=25)
plt.title('Cancellation Rate Across Lead Time Horizons', fontsize=14, fontweight='bold')
plt.ylabel('Cancellation Rate (%)')
plt.ylim(0, 100)
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height() + 2),
                ha='center', fontweight='bold')
plt.show()

### 3.3 The Deposit Paradox & Market Segments

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Deposit Type
dep_rate = df_featured.groupby('deposit_type')['is_canceled'].mean() * 100
dep_rate.plot(kind='bar', ax=axes[0], color=['#10b981', '#ef4444', '#f59e0b'], rot=0)
axes[0].set_title('Deposit Type Cancellation Rate', fontweight='bold')
axes[0].set_ylabel('Cancellation Rate (%)')

# Market Segment
seg_rate = df_featured.groupby('market_segment')['is_canceled'].mean().sort_values() * 100
seg_rate.plot(kind='barh', ax=axes[1], color='#0ea5e9')
axes[1].set_title('Market Segment Cancellation Rate', fontweight='bold')
axes[1].set_xlabel('Cancellation Rate (%)')

plt.tight_layout()
plt.show()

## 4. Machine Learning Model Training & Evaluation
- Target: `is_canceled`
- Zero Data Leakage: Strictly dropped `reservation_status` and `reservation_status_date`.

In [ ]:
from src.model_trainer import train_and_evaluate_all

best_pipeline, metadata = train_and_evaluate_all('hotel_bookings.csv', 'models')

# Display Benchmark Comparison
benchmark_df = pd.DataFrame(metadata['models_benchmark']).T[['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'train_time_seconds']]
benchmark_df.columns = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC', 'Train Time (s)']
display(benchmark_df)

## 5. Live Simulation & Operational Prediction Engine

In [ ]:
from src.predictor import BookingCancellationPredictor

predictor = BookingCancellationPredictor('models/best_cancellation_model.joblib')

# Test booking simulation
sample_booking = {
    'hotel': 'City Hotel',
    'lead_time': 120,
    'arrival_date_month': 'August',
    'stays_in_weekend_nights': 1,
    'stays_in_week_nights': 3,
    'adults': 2,
    'market_segment': 'Online TA',
    'deposit_type': 'No Deposit',
    'adr': 115.0,
    'total_of_special_requests': 0
}

result = predictor.predict_booking(sample_booking)
print(f"Cancellation Probability: {result['cancellation_probability']}%")
print(f"Risk Tier: {result['risk_tier']}")
print(f"Key Drivers: {result['key_drivers']}")
print(f"Recommended Actions: {result['recommended_actions']}")